# Content-Based Recommendation System for KuaiRec Dataset

This notebook implements a content-based filtering approach for video recommendations using the KuaiRec dataset. Content-based filtering recommends items based on their features and characteristics rather than user interactions. In this implementation, we'll use video metadata (captions, categories, etc.) to create feature vectors for each video and recommend similar videos based on these features.

## Stages of Implementation:
1. Data Loading and Pre-processing
2. Feature Engineering and Extraction
3. Creating Content-Based Filters
4. Similarity Computation
5. Building the Recommendation Engine
6. Evaluation and Analysis
7. Discussion and Improvements

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# set plot size
plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

In [ ]:
train_data = pd.read_csv("dataset/kuairec/data/big_matrix.csv")

test_data = pd.read_csv("dataset/kuairec/data/small_matrix.csv")

categories = pd.read_csv("dataset/kuairec/data/item_categories.csv")

In [ ]:
train_data.dropna(inplace=True)
test_data.dropna(inplace=True)
train_data.drop_duplicates(inplace=True)
test_data.drop_duplicates(inplace=True)
train_data = train_data[train_data["timestamp"] >= 0]
test_data = test_data[test_data["timestamp"] >= 0]

In [ ]:
test_data

In [ ]:
train_data

In [ ]:
# Construct item feature matrix
items = train_data.groupby("video_id")
items = items.agg(
    {
        "user_id": "count",
        "video_duration": "mean",
        "timestamp": "max",
        "watch_ratio": "mean",
        "time": "max",
        "date": "max",
    }
)
items.drop(columns=["user_id"], inplace=True)
items.drop(columns=["timestamp"], inplace=True)
items.drop(columns=["watch_ratio"], inplace=True)
items.drop(columns=["date"], inplace=True)
items

In [ ]:
categories

In [ ]:
# videos represent by their categories
import ast

for i in range (31):
    categories['category_' + str(i)] = 0

categories['feat'] = categories['feat'].apply(ast.literal_eval)
for index, row in categories.iterrows():
    # Get the list of features for the current row
    features = row['feat']

    # Set the corresponding category columns to 1
    for feat in features:
        col_name = f'category_{feat}'
        if col_name in categories.columns:
            categories.at[index, col_name] = 1
categories.drop(columns=['feat'], inplace=True)


In [ ]:
categories

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
item_similarity = cosine_similarity(categories.drop(columns=['video_id']), categories.drop(columns=['video_id']))
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=categories['video_id'],
    columns=categories['video_id']
)
item_similarity_df

In [ ]:
from collections import defaultdict
def generate_top_n_recommendations_from_logs(interactions_df, cosine_sim, video_ids, N=10, min_watch_ratio=0.5):
    video_ids = pd.Index(video_ids)  # FIX HERE

    filtered = interactions_df[interactions_df['watch_ratio'] >= min_watch_ratio]
    user_histories = filtered.groupby('user_id')['video_id'].apply(set).to_dict()

    recommendations = {}

    for user, watched in user_histories.items():
        watched_indices = [video_ids.get_loc(v) for v in watched if v in video_ids]
        if not watched_indices:
            recommendations[user] = []
            continue

        sim_scores = np.array(np.sum(cosine_sim[watched_indices], axis=0)).ravel()

        for v in watched:
            if v in video_ids:
                sim_scores[video_ids.get_loc(v)] = -1

        top_indices = np.argsort(sim_scores)[::-1][:N]
        top_videos = video_ids[top_indices].tolist()
        recommendations[user] = top_videos

    return recommendations


In [ ]:


recs = generate_top_n_recommendations_from_logs(
    train_data,
    item_similarity,
    categories['video_id'].reset_index(drop=True),  # optional safety
    N=10,
    min_watch_ratio=0.5
)

# Check the type of the recommendations variable
print(f"Type of recs: {type(recs)}")
print(f"Number of users with recommendations: {len(recs)}")
# Display first 3 recommendations if available
if len(recs) > 0:
    first_user = list(recs.keys())[0]
    print(f"Example recommendations for user {first_user}: {recs[first_user][:5]}...")

In [ ]:
# Build ground truth from test data - using the same threshold as training (0.5)
THRESHOLD = 0.75
high_watch_test = test_data[test_data['watch_ratio'] >= THRESHOLD]
test_truth = high_watch_test.groupby('user_id')['video_id'].apply(set).to_dict()

In [ ]:
N = 10
recs_train_category = generate_top_n_recommendations_from_logs(
    interactions_df=train_data,
    cosine_sim=item_similarity,
    # video_ids=categories['video_id'].reset_index(drop=True),
    video_ids=pd.Index(train_data['video_id'].unique()),
    N=N,
    min_watch_ratio=0.75
)

In [ ]:
def prepare_test_ground_truth(interactions_df, min_watch_ratio=0.75):
    """
    Return user -> set of relevant video_ids from test interactions.
    """
    filtered = interactions_df[interactions_df['watch_ratio'] >= min_watch_ratio]
    return filtered.groupby('user_id')['video_id'].apply(set).to_dict()

test_truth = prepare_test_ground_truth(
    interactions_df=test_data,
    min_watch_ratio=0.75
)


In [ ]:
from sklearn.metrics import ndcg_score
def hit_rate_log(recommendations, test_ground_truth):
    hits, total = 0, 0
    for user, recs in recommendations.items():
        true_items = test_ground_truth.get(user, set())
        hits += len(set(recs) & true_items)
        total += len(true_items)
    return hits / total if total else 0

def precision_at_k_log(recommendations, test_ground_truth, k=10):
    precisions = []
    for user, recs in recommendations.items():
        true_items = test_ground_truth.get(user, set())
        if not true_items:
            continue
        hits = len(set(recs[:k]) & true_items)
        precisions.append(hits / k)
    return np.mean(precisions) if precisions else 0

def ndcg_at_k_log(recommendations, test_ground_truth, k=10):
    ndcgs = []
    for user, recs in recommendations.items():
        true_items = test_ground_truth.get(user, set())
        if not true_items:
            continue
        y_true = [1 if vid in true_items else 0 for vid in recs[:k]]
        y_score = list(range(k, 0, -1))
        ndcgs.append(ndcg_score([y_true], [y_score]))
    return np.mean(ndcgs) if ndcgs else 0

def mrr_log(recommendations, test_ground_truth):
    rr = []
    for user, recs in recommendations.items():
        true_items = test_ground_truth.get(user, set())
        if not true_items:
            continue
        for rank, vid in enumerate(recs, 1):
            if vid in true_items:
                rr.append(1 / rank)
                break
        else:
            rr.append(0)
    return np.mean(rr) if rr else 0


In [ ]:
hit_rate = hit_rate_log(recs_train_category, test_truth)
prec  = precision_at_k_log(recs_train_category, test_truth, k=N)
ndcg = ndcg_at_k_log(recs_train_category, test_truth, k=N)
mrr = mrr_log(recs_train_category, test_truth)
print(f"Hit Rate@{N} for category sim: {hit_rate:.4f}")
print(f"Precision@{N} for category sim: {prec:.4f}")
print(f"NDCG@{N} for category sim: {ndcg:.4f}")
print(f"MRR@{N} for category sim: {mrr:.4f}")
print("---")